# Imports

In [1]:
import os
import glob
import torch
import torchaudio
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Torchmetrics for evaluation
from torchmetrics.classification import BinaryF1Score

In [2]:
# Use GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
# Define paths and dataset info
DATA_DIR = r"c:\Users\Daniel\Deepfake-AI-Voice-Detection-Model\5s_clips\AUDIO_CLEAN"
CLASSES = ["FAKE", "REAL"]

print(f"Classes: {CLASSES}")

Classes: ['FAKE', 'REAL']


# Dataset Class

In [4]:
class AudioDataset(Dataset):
    def __init__(self, data_dir, classes):
        self.file_list = []
        self.labels = []
        for label, class_name in enumerate(classes):
            class_dir = os.path.join(data_dir, class_name)
            if not os.path.exists(class_dir):
                continue
            files = glob.glob(os.path.join(class_dir, "*.mp3")) 
            for f in files:
                self.file_list.append(f)
                self.labels.append(label)
                
        # MelSpectrogram transform
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, n_fft=2048, hop_length=512, n_mels=128
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        
        # Load audio
        waveform, sr = torchaudio.load(file_path)
        
        # Resample if necessary
        if sr != 22050:
            waveform = torchaudio.functional.resample(waveform, sr, 22050)
            
        # Convert stereo to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Pad or truncate to 5 seconds
        target_length = 5 * 22050
        if waveform.shape[1] < target_length:
            waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]
            
        # Create Mel Spectrogram
        mel_spec = self.mel_transform(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)
        
        # Normalize to [0, 1]
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
        
        # Resize for VGG16 (224x224)
        mel_spec_resized = torch.nn.functional.interpolate(
            mel_spec_db.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        # Repeat to 3 channels (RGB) for VGG16
        mel_spec_rgb = mel_spec_resized.repeat(3, 1, 1)
        
        return mel_spec_rgb, torch.tensor(self.labels[idx], dtype=torch.float32)

In [5]:
dataset = AudioDataset(DATA_DIR, CLASSES)
batch_size = 16
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Loaded {len(dataset)} items.")

Loaded 6480 items.


# Architecture of the model

In [9]:
# Load VGG16 with ImageNet weights
vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Freeze base model layers
for param in vgg16.features.parameters():
    param.requires_grad = False

# Modify classifier for our binary classes
num_features = vgg16.classifier[6].in_features
vgg16.classifier[6] = nn.Sequential(
    nn.Linear(num_features, 1),
    nn.Sigmoid()
)

model = vgg16.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total trainable parameters: {trainable_params}")
print(f"Total parameters: {total_params}")

Total trainable parameters: 119549953
Total parameters: 134264641


In [10]:
# Training hyperparameters
num_epochs = 1
learning_rate = 0.001

# Loss function and metric
loss_fn = nn.BCELoss()
# Create optimizer
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    metric = BinaryF1Score().to(device)
    epoch_loss = 0.0
    num_batches = 0
    
    for batch in dataloader:
        # Unpack batch
        inputs_batch, labels_batch = batch
        inputs_batch = inputs_batch.to(device)
        labels_batch = labels_batch.to(device).reshape(-1, 1)
        
        # Zero the gradients
        optimizer.zero_grad()
        # Forward pass - get predictions
        predictions = model(inputs_batch)
        # Compute loss
        loss = loss_fn(predictions, labels_batch)
        # Backward pass
        loss.backward()
        # Update weights
        optimizer.step()
        
        # Track metrics
        epoch_loss += loss.item()
        metric.update(predictions, labels_batch.int())
        num_batches += 1
    
    # Compute metrics at end of epoch
    if num_batches > 0:
        avg_loss = epoch_loss / num_batches
        epoch_f1 = metric.compute().item()
        
        # Print progress every epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, F1-Score: {epoch_f1:.4f}')
    else:
        print("No batches processed. Is the dataset empty?")

print("Training complete!")

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 